In [6]:
from pathlib import Path
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import numpy as np

# ── 1. 尺寸配置 ──────────────────────────────────────────────────────────────

TOTAL_W_MM = 90  # 单个图
TOTAL_H_MM = 100
DPI = 600
W_IN = TOTAL_W_MM / 25.4
H_IN = TOTAL_H_MM / 25.4

# ── 2. 路径配置 ──────────────────────────────────────────────────────────────

HS_PATH = Path("/Volumes/UCL/论文工作/空气污染/HealthServiceIndex.csv")
SHP_PATH = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/city_shp/shi_en.shp")
SHP_CITY_FIELD = "English"
CHINA_SHP = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国底图-中图社审过版本/中国底图/中国面.shp")
CHINA_SHP2 = Path("/Users/shirley/Library/CloudStorage/OneDrive-UniversityCollegeLondon/Desktop/中国国界线/九段线/九段线和群岛.shp")
OUTFILE = Path("/Users/shirley/Desktop/plots_V2/Fig_HS_distribution.png")

PROJ_STR = "+proj=aea +lat_1=25 +lat_2=47 +lat_0=0 +lon_0=105"

# ── 3. 颜色方案（与之前一致）──────────────────────────────────────────────

HS_LEVELS = ["Poor", "Moderate", "Intermediate", "Good", "Excellent", "No data"]
COLOR_SCHEME = {
    "Poor":         "#d7191c",
    "Moderate":     "#f47920",
    "Intermediate": "#f6c101",
    "Good":         "#abd9e9",
    "Excellent":    "#2c7bb6",
    "No data":      "#d3d3d3"  # 灰色表示无数据
}

# ── 4. 读取HS数据 ────────────────────────────────────────────────────────────

hs_df = pd.read_csv(HS_PATH)
hs_df["city"] = hs_df["city"].str.strip()
# 读取所有数据，不做过滤
hs_df = hs_df[["city", "HS_2", "HS_type2"]].copy()

# ── 5. 读取城市shp ───────────────────────────────────────────────────────────

city_shp = gpd.read_file(SHP_PATH).to_crs(PROJ_STR)
city_shp["city_join"] = city_shp[SHP_CITY_FIELD].str.strip().str.lower()

hs_df["city_join"] = hs_df["city"].str.strip().str.lower()

# ── 6. 合并数据 ──────────────────────────────────────────────────────────────

shp_joined = city_shp.merge(hs_df, on="city_join", how="left")

# 处理缺失值，设置category（基于HS_type2）
shp_joined["category"] = shp_joined["HS_type2"].fillna("No data")
shp_joined["category"] = pd.Categorical(
    shp_joined["category"],
    categories=HS_LEVELS,
    ordered=True
)

# ── 7. 转为点（几何中心）─────────────────────────────────────────────────────

shp_points = shp_joined.copy()
shp_points.geometry = shp_points.geometry.centroid

# 分离有数据和无数据的城市（基于HS_type2是否为NA）
dat_with_hs = shp_points[~shp_points["HS_type2"].isna()].copy()
dat_no_hs = shp_points[shp_points["HS_type2"].isna()].copy()

print(f"Total cities: {len(shp_points)}")
print(f"Cities with HS_type2: {len(dat_with_hs)}")
print(f"Cities without HS_type2 (No data): {len(dat_no_hs)}")

# ── 8. 读取底图 ──────────────────────────────────────────────────────────────

china_boundary = gpd.read_file(CHINA_SHP).to_crs(PROJ_STR)
china_boundary2 = gpd.read_file(CHINA_SHP2).to_crs(PROJ_STR)

# 打印边界范围以便调整南海小图
print(f"\n中国边界范围: {china_boundary.total_bounds}")
print(f"九段线边界范围: {china_boundary2.total_bounds}")

# 南海区域边界（根据实际边界调整）
# 通常南海在投影后位于右下角
bounds = china_boundary2.total_bounds
xmin, ymin, xmax, ymax = bounds

# 南海区域大致为整个范围的右下1/4
NANHAI_BOUNDS = (
    xmin + (xmax - xmin) * 0.6,  # x 起点：60%位置
    ymin,                         # y 起点：底部
    xmax,                         # x 终点：右侧
    ymin + (ymax - ymin) * 0.35  # y 终点：35%高度
)

print(f"南海区域范围: {NANHAI_BOUNDS}")

# ── 9. 全局样式 ──────────────────────────────────────────────────────────────

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.titlesize": 8,
    "axes.titleweight": "bold",
})

# ── 10. 绘图函数 ─────────────────────────────────────────────────────────────

def plot_hs_map(ax, add_inset=True):
    """绘制HS分布地图"""
    # 绘制底图
    china_boundary.plot(ax=ax, facecolor="white", edgecolor="black", linewidth=0.3)
    china_boundary2.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.3)

    # 先画无数据的城市（灰色小点）
    if len(dat_no_hs) > 0:
        dat_no_hs.plot(
            ax=ax,
            color=COLOR_SCHEME["No data"],
            markersize=8,
            alpha=0.5,
            edgecolor='none',
            zorder=2
        )

    # 再画有数据的城市（彩色，固定大小）
    if len(dat_with_hs) > 0:
        # 按category分组绘制，所有点大小都是8
        for cat in ["Poor", "Moderate", "Intermediate", "Good", "Excellent"]:
            subset = dat_with_hs[dat_with_hs["category"] == cat]
            if len(subset) > 0:
                subset.plot(
                    ax=ax,
                    color=COLOR_SCHEME[cat],
                    markersize=8,
                    alpha=0.7,
                    edgecolor='none',
                    zorder=3,
                    label=cat
                )
    
    # 添加南海小图
    if add_inset:
        add_nanhai_inset(ax)

def add_nanhai_inset(parent_ax):
    """在地图右下角添加九段线南海小框"""
    x1, y1, x2, y2 = NANHAI_BOUNDS
    
    # 创建小图 (右下角位置)
    axins = parent_ax.inset_axes([0.70, 0.02, 0.28, 0.35])
    axins.set_facecolor("white")
    
    # 绘制底图
    china_boundary.plot(ax=axins, facecolor="white", edgecolor="black", linewidth=0.2)
    china_boundary2.plot(ax=axins, facecolor="none", edgecolor="black", linewidth=0.4)
    
    # 绘制无数据城市
    if len(dat_no_hs) > 0:
        dat_no_hs.plot(
            ax=axins,
            color=COLOR_SCHEME["No data"],
            markersize=2,
            alpha=0.5,
            edgecolor='none',
            zorder=2
        )
    
    # 绘制有数据城市（小图中点更小）
    if len(dat_with_hs) > 0:
        for cat in ["Poor", "Moderate", "Intermediate", "Good", "Excellent"]:
            subset = dat_with_hs[dat_with_hs["category"] == cat]
            if len(subset) > 0:
                subset.plot(
                    ax=axins,
                    color=COLOR_SCHEME[cat],
                    markersize=2,
                    alpha=0.7,
                    edgecolor='none',
                    zorder=3
                )
    
    # 设置南海区域范围
    axins.set_xlim(x1, x2)
    axins.set_ylim(y1, y2)
    
    # 隐藏刻度，保留边框细线
    axins.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in axins.spines.values():
        spine.set_linewidth(0.5)
        spine.set_color("black")

# ── 11. 主图绘制 ─────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(W_IN, H_IN), dpi=DPI)

# 绘制主地图（包含南海小图）
plot_hs_map(ax, add_inset=True)

# ── 12. 图例 ─────────────────────────────────────────────────────────────────

from matplotlib.lines import Line2D

legend_elements = []
for cat in HS_LEVELS:
    # 只显示数据中实际存在的类别
    if cat == "No data":
        if len(dat_no_hs) > 0:
            legend_elements.append(
                Line2D([0], [0], marker='o', color='w', 
                       markerfacecolor=COLOR_SCHEME[cat], 
                       markersize=4, alpha=0.7, 
                       markeredgecolor='none',
                       label=cat)
            )
    else:
        if len(dat_with_hs[dat_with_hs["category"] == cat]) > 0:
            legend_elements.append(
                Line2D([0], [0], marker='o', color='w', 
                       markerfacecolor=COLOR_SCHEME[cat], 
                       markersize=4, alpha=0.7,
                       markeredgecolor='none',
                       label=cat)
            )

ax.legend(
    handles=legend_elements,
    loc='lower left',
    fontsize=5,
    frameon=True,
    facecolor='white',
    edgecolor='none',
    framealpha=0.9,
    markerscale=1.2,
    labelspacing=0.3,
    handletextpad=0.5
)

# ── 13. 样式调整 ─────────────────────────────────────────────────────────────

ax.set_title("Health Service Index Distribution", fontsize=8, fontweight="bold", pad=5)
ax.axis("off")
ax.set_xlim(china_boundary.total_bounds[[0, 2]])
ax.set_ylim(china_boundary.total_bounds[[1, 3]])

# ── 14. 保存 ─────────────────────────────────────────────────────────────────

plt.tight_layout()
OUTFILE.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTFILE, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close(fig)

print(f"✅ Saved → {OUTFILE}")

Total cities: 370
Cities with HS_type2: 293
Cities without HS_type2 (No data): 77

中国边界范围: [-2632237.85592635   410122.34034417  2209177.87072116  5922645.20922108]
九段线边界范围: [ 385758.97551153  362651.82358168 1857616.52212757 2894520.74713675]
南海区域范围: (np.float64(1268873.503481154), np.float64(362651.8235816763), np.float64(1857616.5221275678), np.float64(1248805.946825952))
✅ Saved → /Users/shirley/Desktop/plots_V2/Fig_HS_distribution.png
